In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)

In [ ]:
customers = pd.read_csv("customers.csv")
orders = pd.read_csv("orders.csv")
support = pd.read_csv("support_tickets.csv")
web = pd.read_csv("web_events_snapshot.csv")
labels = pd.read_csv("churn_labels.csv")
interventions = pd.read_csv("intervention_history.csv")

In [ ]:
for name, df in {
    "customers": customers,
    "orders": orders,
    "support": support,
    "web": web,
    "labels": labels,
    "interventions": interventions
}.items():
    print(f"\n{name.upper()}")
    print(df.shape)
    print(df.info())
    print(df.head())

In [ ]:
customers.isnull().mean().sort_values(ascending=False)
orders.isnull().mean().sort_values(ascending=False)

In [ ]:
customers.duplicated().sum()
orders.duplicated().sum()

In [ ]:
orders['customer_id'].isin(customers['customer_id']).mean()
support['customer_id'].isin(customers['customer_id']).mean()

In [ ]:
orders['order_date'] = pd.to_datetime(orders['order_date'])
support['ticket_date'] = pd.to_datetime(support['ticket_date'])

In [ ]:
orders['order_date'].describe()

In [ ]:
df = customers.merge(labels, on='customer_id', how='left')

In [ ]:
labels['churn_next_60d'].value_counts(normalize=True).plot(kind='bar')
plt.title("Churn Distribution")
plt.show()

In [ ]:
orders_agg = orders.groupby('customer_id').agg({
    'order_id':'count',
    'order_value':'sum',
    'order_date':'max'
}).reset_index()

orders_agg.columns = ['customer_id','frequency','monetary','last_order_date']

In [ ]:
snapshot_date = pd.to_datetime("2025-09-30")
orders_agg['recency'] = (snapshot_date - orders_agg['last_order_date']).dt.days

In [ ]:
eda_df = labels.merge(orders_agg, on='customer_id', how='left')

In [ ]:
# Chart 1: Recency vs Churn
sns.boxplot(x='churn_next_60d', y='recency', data=eda_df)
plt.title("Recency vs Churn")
plt.show()

In [ ]:
#Chart 2: Frequency vs Churn
sns.boxplot(x='churn_next_60d', y='frequency', data=eda_df)
plt.show()

In [ ]:
#Chart 3: Monetary vs Churn
sns.boxplot(x='churn_next_60d', y='monetary', data=eda_df)
plt.show()

In [ ]:
#Chart 4: Support Tickets Impact
support_count = support.groupby('customer_id').size().reset_index(name='tickets')
eda_df = eda_df.merge(support_count, on='customer_id', how='left')

sns.boxplot(x='churn_next_60d', y='tickets', data=eda_df)
plt.show()

In [ ]:
#Chart 5: Sentiment vs Churn
sentiment = support.groupby('customer_id')['sentiment_score'].mean().reset_index()
eda_df = eda_df.merge(sentiment, on='customer_id', how='left')

sns.boxplot(x='churn_next_60d', y='sentiment_score', data=eda_df)
plt.show()

In [ ]:
#Chart 6: Web Engagement vs Churn
sns.boxplot(x='churn_next_60d', y='sessions_30d', data=web.merge(labels, on='customer_id'))
plt.show()